# 第05回 知識表現と推論：値とコードを一段ずつ読む

教材内の架空の機器分類・貸出規則を使い、事実、条件、追加された結論を対応付けます。

**開始：** 既存venvのカーネルを選び、初回は変更せず「カーネル再起動→すべて実行」。次に一節ずつ、予想してからShift+Enterで実行します。同梱Pythonのimportは不要で、外部通信も行いません。

**読み方：** 関数の定義は計算方法に名前を付けるセルです。続く短い実行セルが入力を渡します。表・図の表示コードは折り畳み指定です。表示は丸めますが計算には元の値を使います。

**完了の目印：** スマホ→携帯型端末→コンピュータ。基準ではR1・R2、予約を追加するとR3が適用される。


In [ ]:
import sys
print('使用中のPython:',sys.executable)
print('版:',sys.version.split()[0])


In [ ]:
import argparse


In [ ]:
import json


In [ ]:
PARENTS = {
    "携帯型端末": "コンピュータ", "ノートPC": "コンピュータ",
    "据え置き型PC": "コンピュータ", "スマホ": "携帯型端末",
    "タブレット": "携帯型端末", "デスクトップ": "据え置き型PC",
    "サーバ": "据え置き型PC",
}


In [ ]:
FACTS = ("利用登録済み", "講習修了", "在庫あり")


In [ ]:
RULES = (
    ("R1", ("利用登録済み", "講習修了"), "利用資格あり"),
    ("R2", ("利用資格あり", "在庫あり"), "貸出候補"),
    ("R3", ("貸出候補", "予約確認済み"), "貸出準備可"),
)


### 図表の準備（変更不要）

表示用の補助コードです。HTML/SVGを覚える必要はありません。折り畳み表示はNotebook環境によって異なります。


In [ ]:
from html import escape


In [ ]:
from IPython.display import HTML, display


In [ ]:
def learning_format(value):
    if isinstance(value, float):
        return f'{value:.6g}'
    if isinstance(value, (list, tuple)):
        return '['+', '.join(learning_format(item) for item in value)+']'
    if isinstance(value, dict):
        return '{'+', '.join(str(key)+': '+learning_format(item) for key,item in value.items())+'}'
    return str(value)


In [ ]:
def learning_table(title, columns, rows):
    header = ''.join('<th>'+escape(str(c))+'</th>' for c in columns)
    body = ''
    for row in rows:
        body += '<tr>'+''.join('<td>'+escape(learning_format(x))+'</td>' for x in row)+'</tr>'
    return '<div class="scroll"><table><caption>'+escape(title)+'</caption><tr>'+header+'</tr>'+body+'</table></div>'


In [ ]:
def learning_grid(title, matrix):
    rows = [(i,*row) for i,row in enumerate(matrix)]
    return learning_table(title,['行↓ 列→',*range(len(matrix[0]))],rows)


In [ ]:
def learning_plot(title, points):
    # 実計算の値だけを座標へ変換。点の間の値は補間して表示しない。
    xs, ys = [p[0] for p in points], [p[1] for p in points]
    xmin,xmax,ymin,ymax = min(xs),max(xs),min(0,min(ys)),max(ys)
    sx = lambda x: 100+(x-xmin)/max(1e-12,xmax-xmin)*500
    sy = lambda y: 240-(y-ymin)/max(1e-12,ymax-ymin)*190
    marks = ''
    for value in (ymin,(ymin+ymax)/2,ymax):
        marks += f'<path d="M100 {sy(value)} H600" stroke="#aaa"/><text x="0" y="{sy(value)+5}">{value:.4g}</text>'
    for x,y in points:
        marks += f'<circle cx="{sx(x)}" cy="{sy(y)}" r="4" fill="#155e75"><title>{x}: {y}</title></circle>'
    marks += f'<text x="100" y="275">{xmin}</text><text x="580" y="275">{xmax}</text>'
    return '<svg viewBox="0 0 700 310" role="img" aria-label="'+escape(title)+'"><text x="100" y="22">'+escape(title)+'</text>'+marks+'</svg>'


In [ ]:
def learning_show(report, reset=None):
    style = '<style>.learning{font:17px/1.65 sans-serif;color:#172b42;background:white;padding:12px}.learning .scroll{overflow-x:auto}.learning table{border-collapse:collapse;margin:14px 0}.learning td,.learning th{border:1px solid #8394a5;padding:6px 12px;white-space:normal}.learning caption{text-align:left;font-weight:bold}.learning svg{max-width:100%;height:auto}.learning .boards{display:flex;gap:20px;flex-wrap:wrap}.learning .result{border-left:4px solid #155e75;padding:10px}</style>'
    content = '<h3>'+escape(report['title'])+'</h3><p>入力：'+escape(learning_format(report['input']))+'</p>'
    content += learning_table('使う値 → 操作 → 結果',report['columns'],report['rows'])
    for title,columns,rows in report.get('tables',[]):
        content += '<details><summary>詳しい計算：'+escape(title)+'</summary>'+learning_table(title,columns,rows)+'</details>'
    content += '<div class="boards">'+''.join(learning_grid(title,values) for title,values in report.get('matrices',[]))+'</div>'
    for title,points in report.get('plots',[]):
        content += learning_plot(title,points)
    for label in ('result','comparison','explanation','reset'):
        name={'result':'結果','comparison':'一つ変えた比較','explanation':'読み取り方','reset':'戻す'}[label]
        content += '<p class="result">'+name+'：'+escape(reset if label=='reset' and reset is not None else learning_format(report[label]))+'</p>'
    display(HTML(style+'<div class="learning">'+content+'</div>'))


## 1. 分類の経路

**今いる段階：** 用語→親→目標判定

**予想：** 経路がないと何が言えるか

**操作：** example_targetを'コンピュータ'から'ノートPC'へ変更し、同じセルを実行します。

**結果の目印：** コンピュータはyes、ノートPCはunknown

**戻す：** example_target='コンピュータ'に戻して再実行します。

**説明する：** 知識の範囲と否定の違い


### 計算の定義：`isa_path`

単一親の経路を返す。未到達は否定せずunknown、循環はデータの問題。


In [ ]:
def isa_path(parents, start, target):
    """単一親の経路を返す。未到達は否定せずunknown、循環はデータの問題。"""
    known = set(parents) | set(parents.values())
    if start not in known or target not in known:
        return {"status": "unknown", "reason": "unknown_term", "path": []}
    path = [start]
    visited = set()
    current = start
    while True:
        if current == target:
            return {"status": "yes", "reason": "reachable", "path": path}
        if current in visited:
            return {"status": "cycle", "reason": "repeated_node", "path": path}
        visited.add(current)
        if current not in parents:
            return {"status": "unknown", "reason": "no_path", "path": path}
        current = parents[current]
        path.append(current)


### 計算の定義：`lesson_isa`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def lesson_isa(target='コンピュータ'):
    result = isa_path(PARENTS,'スマホ',target)
    rows = [(i,name,'目標に到達' if name==target else '親を調べる' if name in PARENTS else '親の記録なし',PARENTS.get(name,'—')) for i,name in enumerate(result['path'])]
    if not rows:
        rows=[('—','スマホ','探索前に未知語を検出','—')]
    return {'title':'分類の矢印を一つずつたどる','question':'経路がなければ否定してよいか',
            'input':f'スマホは{target}か。教材内の単一親の分類。',
            'columns':['段階','現在の用語','処理','記録された親'],'rows':rows,
            'result':result,'comparison':'コンピュータ→ノートPCでは、経路をたどっても目標へ着かずunknown。',
            'explanation':'スマホ→携帯型端末→コンピュータという分類上の関係。現実の全分類を収録したデータではなく、未到達は否定ではない。',
            'reset':'target="コンピュータ"で再実行。'}


In [ ]:
example_target = 'コンピュータ'
program_observation_isa = lesson_isa(example_target)
learning_show(program_observation_isa, reset="example_target='コンピュータ'に戻して再実行します。")


## 2. 規則で事実を増やす

**今いる段階：** 事実→照合→結論→再照合

**予想：** どの条件が不足するか

**操作：** example_reservationをFalseからTrueへ変更し、同じセルを実行します。

**結果の目印：** R3は予約確認済みが不足

**戻す：** example_reservation=Falseに戻して再実行します。

**説明する：** 追加された結論を次の規則が使う理由


### 計算の定義：`forward_chain`

肯定的な事実を追加する最小推論器。規則順に照合し、追加時だけ記録。


In [ ]:
def forward_chain(initial_facts, rules, record=None):
    """肯定的な事実を追加する最小推論器。規則順に照合し、追加時だけ記録。"""
    facts = set(initial_facts)  # 呼び出し元の事実は変えない
    trace = []
    changed = True
    while changed:
        changed = False
        for rule_id, conditions, conclusion in rules:
            missing = [condition for condition in conditions if condition not in facts]
            already_known = conclusion in facts
            if record is not None:
                record.append((rule_id, sorted(facts), list(conditions), missing, conclusion,
                               '追加済み' if already_known else '条件不足' if missing else '追加する'))
            if not missing and not already_known:
                facts.add(conclusion)
                trace.append({"rule": rule_id, "conditions": list(conditions),
                              "conclusion": conclusion})
                changed = True
    return {"facts": sorted(facts), "trace": trace}


### 計算の定義：`lesson_forward`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def lesson_forward(reservation=False):
    facts = list(FACTS)
    if reservation:
        facts.append('予約確認済み')
    trace = []
    result = forward_chain(facts,RULES,record=trace)
    return {'title':'事実カードと規則の条件を照合する','question':'どの規則で新しい事実が増えたか',
            'input':facts,'columns':['規則','その時点の事実','必要条件','不足条件','結論','判定'],'rows':trace,
            'tables':[('実際に追加した順',['規則','使った条件','追加した事実'],[(r['rule'],r['conditions'],r['conclusion']) for r in result['trace']])],
            'result':result['facts'],'comparison':'予約確認済みを追加するとR3も条件がそろい、貸出準備可が加わる。',
            'explanation':'同じ周回内でも、先の規則で加えた事実を後の規則が使える。追加がなくなるまで繰り返す。規則の適用と単なる照合を区別する。',
            'reset':'reservation=Falseで再実行。FACTSのコピーを使う。'}


In [ ]:
example_reservation = False
program_observation_forward = lesson_forward(example_reservation)
learning_show(program_observation_forward, reset='example_reservation=Falseに戻して再実行します。')


## 3. 不明を否定しない

**今いる段階：** 入力の欠落→適用不可→不明

**予想：** 記録がなければ未修了か

**操作：** example_remove_trainingをTrueからFalseへ変更し、同じセルを実行します。

**結果の目印：** 講習修了なしでは利用資格も導けない

**戻す：** example_remove_training=Trueに戻して再実行します。

**説明する：** 実装している推論の範囲


### 計算の定義：`lesson_missing`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def lesson_missing(remove_training=True):
    facts = [f for f in FACTS if not (remove_training and f=='講習修了')]
    result = forward_chain(facts,RULES)
    rows = [(fact,'記録あり' if fact in result['facts'] else '導けない（不明）') for fact in ['講習修了','利用資格あり','貸出候補','貸出準備可']]
    return {'title':'導けないことと否定を区別する','question':'講習修了が記録にない人は未修了と断定できるか',
            'input':facts,'columns':['調べたい事実','この知識から言えること'],'rows':rows,
            'result':result,'comparison':'講習修了を除くとR1・R2が導けない。戻すと利用資格あり・貸出候補まで導ける。',
            'explanation':'このプログラムは肯定的事実を追加する前向き推論器。明示的な否定・例外の優先順位・後ろ向き探索は実装していない。事実がないことを否定へ置き換えない。',
            'reset':'remove_training=Trueで再実行。元のFACTSは保持。'}


In [ ]:
example_remove_training = True
program_observation_missing = lesson_missing(example_remove_training)
learning_show(program_observation_missing, reset='example_remove_training=Trueに戻して再実行します。')


## 基準結果と既存の変更実験

ここまでの図表を、従来の基準出力と照合します。以下の実験も入力のコピーを使います。定義を変更した場合は再起動して全セル実行してください。


### 計算の定義：`demo`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def demo():
    queries = (("スマホ", "コンピュータ"), ("サーバ", "コンピュータ"),
               ("スマホ", "ノートPC"), ("ENIAC", "コンピュータ"),
               ("スマホ", "電子機器"), ("コンピュータ", "コンピュータ"))
    return {
        "isa": [{"start": a, "target": b, **isa_path(PARENTS, a, b)} for a, b in queries],
        "cycle": isa_path({"A": "B", "B": "A", "C": "D"}, "A", "D"),
        "initial_facts": list(FACTS),
        "baseline": forward_chain(FACTS, RULES),
        "with_reservation": forward_chain((*FACTS, "予約確認済み"), RULES),
        "without_training": forward_chain(tuple(f for f in FACTS if f != "講習修了"), RULES),
    }


## 入力と予想

端末の分類は昨年度isa.pyに基づく教材内の定義です。現実の機器分類を網羅しません。スマホ→携帯型端末→コンピュータをたどると「はい」。経路がない場合は不明とします。

架空の貸出規則は、R1：登録済み＆講習修了→利用資格あり、R2：利用資格あり＆在庫あり→貸出候補、R3：貸出候補＆予約確認済み→貸出準備可です。一件の申請のみを扱います。学校の実際の規則ではありません。

基準入力は登録済み・講習修了・在庫あり。R1、R2が適用され、予約確認がないのでR3は適用されないはずです。外部データ・乱数は使いません。


## 基準と固定比較を実行

経路、最終事実、適用した規則を順に読みます。`baseline_result`は基準の照合用に残し、後の実験で上書きしません。


In [ ]:
import json
baseline_result = demo()
print(json.dumps(baseline_result, ensure_ascii=False, indent=2))


## 出力の読み方

`status`のyesは確認できた経路、unknownは未知語または経路なし、cycleは未到達のまま同じノードに戻った状態です。既知の同一概念は長さ0の経路で確認済みとします。未知語同士の同一名は不明です。

`baseline`ではR1→R2、`with_reservation`ではR1→R2→R3、`without_training`では適用規則なしになります。factsは比較しやすい文字列順で、推論の順序ではありません。推論の順序はtraceを読みます。

循環例A→B→Aは有限回で止まります。循環より先に目標へ到達した場合は経路ありです。知識全体の無循環性を検査する機能ではありません。


## 自分で一つ変える：予約確認を追加

まず結論を予想し、次のセルを実行します。初期事実のコピーに一つ追加するので、元のFACTSは変わりません。


In [ ]:
trial_facts = set(FACTS)
trial_facts.add("予約確認済み")
trial_result = forward_chain(trial_facts, RULES)
print(json.dumps(trial_result, ensure_ascii=False, indent=2))
print("元の事実:", FACTS)


## 自分で一つ変える：知識を追加

電子機器は初めは未登録です。コピーに「コンピュータ→電子機器」を追加してから、同じ問いを試します。この関係も教材内で与える定義です。


In [ ]:
trial_parents = dict(PARENTS)
print("追加前:", isa_path(PARENTS, "スマホ", "電子機器"))
trial_parents["コンピュータ"] = "電子機器"
print("追加後:", isa_path(trial_parents, "スマホ", "電子機器"))
print("元の知識は保持:", "コンピュータ" not in PARENTS)


## 振り返り

- どのリンク、またはどの条件が結果を変えましたか。
- 導けない結論を、なぜ否定と断定しないのでしょうか。
- R3、R2、R1の順に規則を並べたら、一巡で終わるでしょうか。

必要なら別セルでコピーを作って確かめます。後ろ向き推論・フレームの既定値・明示的否定・複数申請・RETEはこのプログラムには実装していません。


## 振り返りと復帰

入力・行った処理・結果・変更前との差を、自分の言葉で説明してください。観察セルは毎回基準のコピーから計算するため、同じセルを再実行しても前の実験の続きにはなりません。NameErrorや実行順が不明な場合はカーネル再起動→全セル実行。元の配布ファイルを残し、変更版は別名保存します。

Python版は`python knowledge_reasoning.py --explain all`で同じ計算表を表示します。各テーマは`--explain isa`のように指定できます。従来のコマンドと固定期待値も維持しています。
